In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

# Hyperparameters
GAMMA = 0.99
LR = 0.0003
EPS_CLIP = 0.2
K_EPOCH = 10
T_HORIZON = 20

class BitcoinTradingEnv:
    def __init__(self, price_data):
        self.price_data = price_data
        self.current_step = 0
        self.balance = 10000  # 초기 자본
        self.position = 0  # 현재 포지션
        self.done = False
    
    def reset(self):
        self.current_step = 0
        self.balance = 10000
        self.position = 0
        self.done = False
        return self._get_state()
    
    def _get_state(self):
        return np.array([self.balance, self.position, self.price_data[self.current_step]])
    
    def step(self, action):
        if self.done:
            return self._get_state(), 0, True
        
        prev_price = self.price_data[self.current_step]
        self.current_step += 1
        if self.current_step >= len(self.price_data) - 1:
            self.done = True
        
        current_price = self.price_data[self.current_step]
        reward = 0
        
        if action == 1:  # 매수
            self.position = self.balance / current_price
            self.balance = 0
        elif action == 2:  # 매도
            self.balance = self.position * current_price
            self.position = 0
        
        if self.done:
            self.balance += self.position * current_price
            self.position = 0
        reward = self.balance + (self.position * current_price) - 10000
        
        return self._get_state(), reward, self.done

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.fc1 = nn.Linear(state_dim, 256)
        self.fc_pi = nn.Linear(256, action_dim)
        self.fc_v = nn.Linear(256, 1)
    
    def pi(self, x):
        x = torch.relu(self.fc1(x))
        return torch.softmax(self.fc_pi(x), dim=-1)
    
    def v(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc_v(x)

class PPO:
    def __init__(self, state_dim, action_dim):
        self.model = ActorCritic(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=LR)
        self.data = []
    
    def put_data(self, transition):
        self.data.append(transition)
    
    def train(self):
        s, a, r, s_prime, prob_a, done_mask = zip(*self.data)
        
        s = torch.tensor(s, dtype=torch.float)
        a = torch.tensor(a, dtype=torch.long).unsqueeze(1)
        r = torch.tensor(r, dtype=torch.float).unsqueeze(1)
        s_prime = torch.tensor(s_prime, dtype=torch.float)
        prob_a = torch.tensor(prob_a, dtype=torch.float).unsqueeze(1)
        done_mask = torch.tensor(done_mask, dtype=torch.float).unsqueeze(1)
        
        for _ in range(K_EPOCH):
            pi = self.model.pi(s)
            pi_a = pi.gather(1, a)
            v = self.model.v(s)
            v_prime = self.model.v(s_prime)
            
            td_target = r + GAMMA * v_prime * done_mask
            delta = td_target - v
            
            advantage = delta.detach()
            ratio = torch.exp(torch.log(pi_a) - torch.log(prob_a))
            surr1 = ratio * advantage
            surr2 = torch.clamp(ratio, 1 - EPS_CLIP, 1 + EPS_CLIP) * advantage
            
            loss = -torch.min(surr1, surr2) + (v - td_target).pow(2)
            
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()
        
        self.data = []

# Simulated Price Data
price_data = np.sin(np.linspace(0, 100, 1000)) * 100 + 50000  # 가상 비트코인 가격

# Training Loop
env = BitcoinTradingEnv(price_data)
state_dim = 3
action_dim = 3  # [0: 유지, 1: 매수, 2: 매도]
ppo = PPO(state_dim, action_dim)

for n_epi in range(1000):
    state = env.reset()
    done = False
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float)
        prob = ppo.model.pi(state_tensor)
        m = Categorical(prob)
        action = m.sample().item()
        print(action)
        next_state, reward, done = env.step(action)
        ppo.put_data((state, action, reward, next_state, prob[action].item(), 1 - done))
        
        state = next_state
    
    ppo.train()
    
    if n_epi % 100 == 0:
        print(f"Episode {n_epi}, Balance: {env.balance}, Position Value: {env.position * env.price_data[env.current_step]}")


tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tensor(0)
0
tens

KeyboardInterrupt: 